## [프로젝트] Seq2Seq으로 한국어 번역기 만들기

In [1]:
!pip install torch
!pip install mecab-python3

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import re
import MeCab
import time

# ==========================================
# GPU/CPU 디바이스 설정
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cuda


In [3]:
# ==========================================
# Step 1 & 2: 데이터 파일 로드 및 정제
# ==========================================
print("데이터 로드 및 정제 시작...")

path_ko = 'work/korean-english-park.train/korean-english-park.train.ko'
path_en = 'work/korean-english-park.train/korean-english-park.train.en'

with open(path_ko, 'r', encoding='utf-8') as f:
    ko_raw = f.read().splitlines()
with open(path_en, 'r', encoding='utf-8') as f:
    en_raw = f.read().splitlines()

# 중복 제거 (set 활용, 병렬성 유지)
raw_corpus = list(set(zip(ko_raw, en_raw)))
print(f"중복 제거 후 데이터 수: {len(raw_corpus)}")

def preprocess_sentence(sentence, is_english=False):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Zㄱ-ㅎ가-힣?.!,]+", " ", sentence)
    sentence = sentence.strip()
    
    if is_english:
        sentence = '<start> ' + sentence + ' <end>'
    return sentence

mecab = MeCab.Tagger()
kor_corpus = []
eng_corpus = []

for ko, en in raw_corpus:
    ko_pre = preprocess_sentence(ko, is_english=False)
    en_pre = preprocess_sentence(en, is_english=True)
    
    parsed = mecab.parse(ko_pre)
    ko_tokens = [line.split('\t')[0] for line in parsed.split('\n') if line not in ['EOS', '']]
    en_tokens = en_pre.split()
    
    if len(ko_tokens) <= 40 and len(en_tokens) <= 40:
        kor_corpus.append(" ".join(ko_tokens))
        eng_corpus.append(en_pre)

print(f"길이 40 이하 필터링 후 데이터 수: {len(kor_corpus)}")

데이터 로드 및 정제 시작...
중복 제거 후 데이터 수: 78968
길이 40 이하 필터링 후 데이터 수: 62733


In [5]:
# ==========================================
# Step 3: 데이터 토큰화 및 텐서 변환 (Custom Vocab)
# ==========================================
class Vocabulary:
    def __init__(self, num_words=15000):
        # 0: 패딩, 1: OOV(Unknown)
        self.word2idx = {"<pad>": 0, "<unk>": 1}
        self.idx2word = {0: "<pad>", 1: "<unk>"}
        self.num_words = num_words
        self.idx = 2
        
    def fit(self, sentences):
        word_counts = {}
        for sentence in sentences:
            for word in sentence.split():
                word_counts[word] = word_counts.get(word, 0) + 1
                
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        
        for word, _ in sorted_words:
            if len(self.word2idx) >= self.num_words:
                break
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def texts_to_sequences(self, sentences):
        return [[self.word2idx.get(word, 1) for word in sentence.split()] for sentence in sentences]

def pad_sequences(sequences, padding='post'):
    max_len = max(len(seq) for seq in sequences)
    padded = np.zeros((len(sequences), max_len), dtype=np.int64) # PyTorch는 CrossEntropy에 int64(LongTensor) 요구
    for i, seq in enumerate(sequences):
        if padding == 'post':
            padded[i, :len(seq)] = seq
        else:
            padded[i, -len(seq):] = seq
    return padded

# 토크나이저 훈련 및 텐서 변환
NUM_WORDS = 15000

ko_vocab = Vocabulary(NUM_WORDS)
ko_vocab.fit(kor_corpus)
ko_seqs = ko_vocab.texts_to_sequences(kor_corpus)
ko_tensor = pad_sequences(ko_seqs, padding='post')

en_vocab = Vocabulary(NUM_WORDS)
en_vocab.fit(eng_corpus)
en_seqs = en_vocab.texts_to_sequences(eng_corpus)
en_tensor = pad_sequences(en_seqs, padding='post')

SRC_VOCAB_SIZE = len(ko_vocab.word2idx)
TGT_VOCAB_SIZE = len(en_vocab.word2idx)
print(f"한국어 단어 사전 크기: {SRC_VOCAB_SIZE}")
print(f"영어 단어 사전 크기: {TGT_VOCAB_SIZE}")

한국어 단어 사전 크기: 15000
영어 단어 사전 크기: 15000


In [6]:
# ==========================================
# Step 4: 모델 설계 (Bahdanau Attention 기반 Seq2seq)
# ==========================================
BATCH_SIZE = 64
EMBEDDING_DIM = 256
HIDDEN_UNITS = 512

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, enc_units, batch_first=True)

    def forward(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, hidden)
        return output, state

class BahdanauAttention(nn.Module):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = nn.Linear(units, units)
        self.W2 = nn.Linear(units, units)
        self.V = nn.Linear(units, 1)

    def forward(self, hidden, enc_output):
        # hidden shape: (1, batch_size, hidden_size) -> (batch_size, 1, hidden_size)
        hidden_with_time_axis = hidden.permute(1, 0, 2)
        
        score = self.V(torch.tanh(self.W1(enc_output) + self.W2(hidden_with_time_axis)))
        attention_weights = torch.softmax(score, dim=1)
        
        context_vector = attention_weights * enc_output
        context_vector = torch.sum(context_vector, dim=1)
        return context_vector, attention_weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim + dec_units, dec_units, batch_first=True)
        self.fc = nn.Linear(dec_units, vocab_size)
        self.attention = BahdanauAttention(dec_units)

    def forward(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        
        x = self.embedding(x)
        x = torch.cat((context_vector.unsqueeze(1), x), dim=-1)
        
        output, state = self.gru(x, hidden)
        output = output.squeeze(1)
        x = self.fc(output)
        
        return x, state, attention_weights

encoder = Encoder(SRC_VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_UNITS).to(device)
decoder = Decoder(TGT_VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_UNITS).to(device)

In [8]:
# ==========================================
# Step 5: 훈련하기 및 테스트
# ==========================================

import torch

# 이상 탐지 모드를 켜면 nan이 발생하는 정확한 레이어와 원인을 에러 메시지로 띄워줍니다.
torch.autograd.set_detect_anomaly(False)

# 패딩(0)은 손실 계산에서 제외
criterion = nn.CrossEntropyLoss(ignore_index=0) 
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.0005)

# DataLoader 생성
dataset = TensorDataset(torch.tensor(ko_tensor), torch.tensor(en_tensor))
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

def train_step(inp, targ):
    loss = 0
    optimizer.zero_grad()
    
    # 초기 hidden state
    enc_hidden = torch.zeros(1, BATCH_SIZE, HIDDEN_UNITS).to(device)
    
    enc_output, enc_hidden = encoder(inp, enc_hidden)
    dec_hidden = enc_hidden
    
    # 디코더의 첫 입력은 <start> 토큰
    dec_input = torch.tensor([[en_vocab.word2idx['<start>']] * BATCH_SIZE]).squeeze(0).unsqueeze(1).to(device)
    
    # Teacher Forcing
    for t in range(1, targ.size(1)):
        predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
        # 현재 타임스텝의 타겟이 모두 패딩(0)이 아닐 때만 Loss 누적
        if (targ[:, t] != 0).sum() > 0:
            loss += criterion(predictions, targ[:, t])
        dec_input = targ[:, t].unsqueeze(1) # 다음 스텝의 입력은 실제 정답 타겟
        
    batch_loss = loss.item() / int(targ.size(1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
    optimizer.step()
    
    return batch_loss

def translate(sentence):
    encoder.eval()
    decoder.eval()
    
    sentence = preprocess_sentence(sentence, is_english=False)
    parsed = mecab.parse(sentence)
    tokens = [line.split('\t')[0] for line in parsed.split('\n') if line not in ['EOS', '']]
    
    inputs = [ko_vocab.word2idx.get(i, ko_vocab.word2idx['<unk>']) for i in tokens]
    # 모델 추론 시 배치 사이즈는 1
    inputs = torch.tensor(inputs).unsqueeze(0).to(device)
    
    result = ''
    hidden = torch.zeros(1, 1, HIDDEN_UNITS).to(device)
    
    with torch.no_grad():
        enc_out, enc_hidden = encoder(inputs, hidden)
        dec_hidden = enc_hidden
        dec_input = torch.tensor([[en_vocab.word2idx['<start>']]]).to(device)
        
        max_len = en_tensor.shape[1]
        
        for t in range(max_len):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_out)
            predicted_id = predictions.argmax(1).item()
            
            if en_vocab.idx2word[predicted_id] == '<end>':
                result += ' <end>'
                return result.strip()
            
            result += en_vocab.idx2word[predicted_id] + ' '
            dec_input = torch.tensor([[predicted_id]]).to(device)
            
    return result.strip()

EPOCHS = 10 
examples = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

print("\n본격적인 학습을 시작합니다!")
for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0
    
    encoder.train()
    decoder.train()
    
    for batch_idx, (inp, targ) in enumerate(dataloader):
        inp, targ = inp.to(device), targ.to(device)
        batch_loss = train_step(inp, targ)
        total_loss += batch_loss

    print(f'Epoch {epoch + 1} Loss {total_loss / len(dataloader):.4f}')
    print(f'Time taken for 1 epoch {time.time() - start:.2f} sec\n')
    
    for i, ex in enumerate(examples):
        print(f"K{i+1}) {ex}")
        print(f"E{i+1}) {translate(ex)}")
    print("-" * 50)


본격적인 학습을 시작합니다!
Epoch 1 Loss 4.3741
Time taken for 1 epoch 188.61 sec

K1) 오바마는 대통령이다.
E1) obama is the first time in the democratic nomination .  <end>
K2) 시민들은 도시 속에 산다.
E2) the people were not to be <unk> .  <end>
K3) 커피는 필요 없다.
E3) it s not clear .  <end>
K4) 일곱 명의 사망자가 발생했다.
E4) the people were killed in the town of the country s death toll , the military said .  <end>
--------------------------------------------------
Epoch 2 Loss 3.8588
Time taken for 1 epoch 188.40 sec

K1) 오바마는 대통령이다.
E1) obama s president barack obama is the first time in the president , and the president is not .  <end>
K2) 시민들은 도시 속에 산다.
E2) they are not <unk> <unk> .  <end>
K3) 커피는 필요 없다.
E3) the <unk> was not going to be a little bit .  <end>
K4) 일곱 명의 사망자가 발생했다.
E4) the quake were killed in the past two days .  <end>
--------------------------------------------------
Epoch 3 Loss 3.4902
Time taken for 1 epoch 189.07 sec

K1) 오바마는 대통령이다.
E1) obama is the president elect barack obama is to be a president 

### 회고  

##### - Epoch를 거듭할수록 Loss는 4.37에서 1.91까지 감소하였으나, 아직 번역 퀄리티가 좋아지지는 않은 것 같다.  
##### 학습 횟수를 50회 이상 늘리게 되면 학습 효과가 더 좋아지겠지만 1Epoch당 3분씩 소요되기에 이 부분은 진행 해 보지 못하였다.  
##### - 다른 대안으로 **Teacher Forcing 비율 점진적 감소** 나 **양방향 인코더(Bidirectional GRU)** 를 적용하는 대안을 Ai가 제시하였으나,  
시간 관계 상 여기까지는 진행하지 못한 게 아쉬움으로 남는다.  

